In [14]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    GPT2LMHeadModel,
    GPT2Config,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorForLanguageModeling,
)
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
import re
import numpy as np
from datasets import Dataset, disable_progress_bar
import wandb

disable_progress_bar()

In [ ]:
def augment_route_for_set_learning(
    frames: str, tokenizer: PreTrainedTokenizerFast, num_augmentations: int = 3
) -> List[Dict[str, List[int]]]:
    """
    Create multiple training examples from a single route with different orderings.

    This teaches the model that holds belong together as a SET, not a SEQUENCE.

    Args:
        frames: Original route string (e.g., "angle40 grade18 start1 hand2 finish3")
        tokenizer: Tokenizer to encode the route
        num_augmentations: Number of different examples to create

    Returns:
        List of dicts with 'prefix' and 'remaining' keys
    """
    tokens = tokenizer.encode(frames, add_special_tokens=True)
    bos, eos = tokens[0], tokens[-1]
    content_tokens = tokens[1:-1]

    if len(content_tokens) < 2:
        return [{"prefix": [bos], "remaining": tokens[1:]}]

    metadata = content_tokens[:2]  # angle and grade
    holds = content_tokens[2:]  # all holds
    augmented_examples = []

    # AUGMENTATION 1: Empty prompt (just BOS) → predict everything (5% chance)
    if np.random.random() < 0.05:
        augmented_examples.append(
            {"prefix": [bos], "remaining": metadata + holds + [eos]}
        )

    # AUGMENTATION 2: Metadata only → predict all holds + EOS (10% chance)
    if np.random.random() < 0.10 and len(holds) > 0:
        augmented_examples.append(
            {"prefix": [bos] + metadata, "remaining": holds + [eos]}
        )

    # AUGMENTATION 3-N: Random orderings with random split points
    for _ in range(num_augmentations):
        if len(holds) == 0:
            continue

        shuffled_holds = holds.copy()
        np.random.shuffle(shuffled_holds)

        split_idx = (
            np.random.randint(0, len(shuffled_holds)) if len(shuffled_holds) > 1 else 0
        )
        prefix_holds = shuffled_holds[:split_idx]
        remaining_holds = shuffled_holds[split_idx:]

        augmented_examples.append(
            {
                "prefix": [bos] + metadata + prefix_holds,
                "remaining": remaining_holds + [eos],
            }
        )

    return augmented_examples


def create_augmented_dataset(
    dataset: Dataset, tokenizer: PreTrainedTokenizerFast, num_augmentations: int = 3
) -> Dataset:
    """
    Transform a dataset of routes into an augmented dataset for set-based learning.
    Each original route becomes multiple training examples with different orderings.
    """
    all_prefixes = []
    all_remaining = []

    for example in dataset:
        augmented = augment_route_for_set_learning(
            example["frames"], tokenizer, num_augmentations
        )
        for aug_example in augmented:
            all_prefixes.append(aug_example["prefix"])
            all_remaining.append(aug_example["remaining"])

    return Dataset.from_dict({"prefix": all_prefixes, "remaining": all_remaining})


In [16]:
def preprocess_datasets_for_set_learning(
    datasets: Dict[str, Dataset],
    tokenizer: PreTrainedTokenizerFast,
    num_augmentations: int = 3,
    ) -> Dict[str, Dataset]:
    """
    Preprocess datasets for set-based learning.

    - TRAIN: Shuffled augmentation (teaches set membership, order-invariance)
             Uses custom prefix/remaining format
    - VAL/TEST: Standard tokenization (no augmentation, standard GPT evaluation)
                Uses standard input_ids format for DataCollatorForLanguageModeling
    """
    processed_datasets = {}

    processed_datasets["train"] = create_augmented_dataset(
        datasets["train"], tokenizer, num_augmentations=num_augmentations
    )

    processed_datasets["val"] = datasets["val"].map(
        lambda example: tokenizer(example["frames"]),
        batched=True,
        remove_columns=datasets["val"].column_names,
    )

    processed_datasets["test"] = datasets["test"].map(
        lambda example: tokenizer(example["frames"]),
        batched=True,
        remove_columns=datasets["test"].column_names,
    )

    return processed_datasets

In [17]:
@dataclass
class SetBasedDataCollator:
    """
    Collates batches for set-based prediction.

    Key difference from standard collators:
    - 'remaining' stays as List[List[int]] (not padded tensor)
    - Only 'prefix' (input_ids) is padded
    """

    tokenizer: PreTrainedTokenizerFast

    def __call__(self, examples: List[Dict]) -> Dict:
        prefixes = [ex["prefix"] for ex in examples]
        remaining_sets = [ex["remaining"] for ex in examples]
        max_prefix_len = max(len(p) for p in prefixes)

        input_ids = []
        attention_mask = []

        for prefix in prefixes:
            padding_length = max_prefix_len - len(prefix)
            padded_ids = prefix + [self.tokenizer.pad_token_id] * padding_length
            mask = [1] * len(prefix) + [0] * padding_length
            input_ids.append(padded_ids)
            attention_mask.append(mask)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "remaining": remaining_sets,  # Keep as list of lists!
        }



In [18]:
class ShuffleKilterGPT(nn.Module):
    """
    GPT-2 model for set-based climbing route generation.

    Key differences from sequential model:
    1. Learns which holds belong together (set membership)
    2. Loss computed only at last position of prefix
    3. Predicts over "remaining tokens" not "next token in sequence"
    """

    def __init__(
        self,
        vocab_size: int,
        n_embd: int = 256,
        n_head: int = 4,
        n_layer: int = 6,
        n_positions: int = 128,
        dropout: float = 0.1,
        ):
        super().__init__()
        config = GPT2Config(
            vocab_size=vocab_size,
            n_embd=n_embd,
            n_head=n_head,
            n_layer=n_layer,
            n_positions=n_positions,
            n_ctx=n_positions,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
        )
        self.model = GPT2LMHeadModel(config)
        self.config = config

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        remaining: Optional[List[List[int]]] = None,
        ):
        """
        Forward pass with set-based loss.

        Args:
            input_ids: [batch_size, seq_len] - prefix tokens
            attention_mask: [batch_size, seq_len] - attention mask
            remaining: List of lists - valid next tokens for each example

        Returns:
            Model outputs with custom loss
        """
        outputs = self.model(
            input_ids=input_ids, attention_mask=attention_mask, labels=None
        )

        if remaining is not None:
            logits = outputs.logits
            batch_size = logits.size(0)

            # Find last non-padded position for each example
            if attention_mask is not None:
                seq_lengths = attention_mask.sum(dim=1) - 1
            else:
                seq_lengths = torch.full(
                    (batch_size,), logits.size(1) - 1, device=logits.device
                )

            # Extract logits at last position only
            last_logits = logits[
                torch.arange(batch_size, device=logits.device), seq_lengths
            ]
            log_probs = F.log_softmax(last_logits, dim=-1)

            # Multi-label loss: log-sum-exp over valid tokens
            loss = 0.0
            for i in range(batch_size):
                if len(remaining[i]) == 0:
                    continue

                valid_tokens = torch.tensor(
                    remaining[i], dtype=torch.long, device=logits.device
                )
                valid_log_probs = log_probs[i, valid_tokens]

                # Loss = -log P(any valid token) = -logsumexp(log probs of valid tokens)
                loss -= torch.logsumexp(valid_log_probs, dim=0)

            outputs.loss = loss / batch_size

        return outputs

    def generate_route(
        self,
        tokenizer: PreTrainedTokenizerFast,
        angle: int = 40,
        grade: int = 18,
        max_holds: int = 20,
        temperature: float = 1.0,
        top_p: float = 0.95,
        device: str = "cpu",
        required_holds: Optional[List[str]] = None,
        ) -> str:
        """
        Generate a route using iterative set-based sampling.

        Unlike standard autoregressive generation, this:
        1. Samples from distribution over remaining tokens
        2. Prevents duplicates
        3. Can enforce constraints (required holds)

        Args:
            tokenizer: Tokenizer
            angle: Board angle (20-60, multiples of 5)
            grade: Route grade (13-27)
            max_holds: Maximum number of holds to generate
            temperature: Sampling temperature
            top_p: Nucleus sampling threshold
            device: Device to run on
            required_holds: Optional list of hold names that must be included

        Returns:
            Generated route string
        """
        self.model.eval()
        self.model.to(device)

        angle_rounded = max(20, min(60, round(angle / 5) * 5))
        grade = max(13, min(27, grade))

        angle_str = f"angle{angle_rounded}"
        grade_str = f"grade{grade}"
        angle_token = tokenizer.encode(angle_str, add_special_tokens=False)[0]
        grade_token = tokenizer.encode(grade_str, add_special_tokens=False)[0]

        prefix = [tokenizer.bos_token_id, angle_token, grade_token]
        used_tokens = set(prefix)

        # Handle required holds
        required_token_ids = []
        if required_holds:
            for hold in required_holds:
                hold_tokens = tokenizer.encode(hold, add_special_tokens=False)
                if len(hold_tokens) > 0:
                    required_token_ids.append(hold_tokens[0])

        # Generation loop
        for _ in range(max_holds):
            input_ids = torch.tensor([prefix], device=device)
            attention_mask = torch.ones_like(input_ids)

            with torch.no_grad():
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits[0, -1, :] / temperature

                # Mask already-used tokens
                for token in used_tokens:
                    logits[token] = float("-inf")

                probs = F.softmax(logits, dim=-1)

                # Nucleus sampling (top-p)
                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumsum_probs = torch.cumsum(sorted_probs, dim=0)
                sorted_indices_to_remove = cumsum_probs > top_p
                sorted_indices_to_remove[0] = False

                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                probs[indices_to_remove] = 0.0

                # Renormalize
                if probs.sum() > 0:
                    probs = probs / probs.sum()
                else:
                    probs = torch.ones_like(probs)
                    for token in used_tokens:
                        probs[token] = 0.0
                    probs = probs / probs.sum()

                next_token = torch.multinomial(probs, num_samples=1).item()

                if next_token == tokenizer.eos_token_id:
                    break

                prefix.append(next_token)
                used_tokens.add(next_token)

        # Ensure required holds are present
        if required_token_ids:
            for req_token in required_token_ids:
                if req_token not in used_tokens:
                    prefix.insert(-1, req_token)

        return tokenizer.decode(prefix, skip_special_tokens=True)

    def validate_route(self, route_str: str) -> Tuple[bool, str]:
        """Validate that a generated route meets constraints."""
        pattern = r"angle(\d+)\s+grade(\d+)\s+(.*)"
        match = re.match(pattern, route_str)

        if not match:
            return False, "Invalid route format"

        angle, grade, holds_str = match.groups()
        angle, grade = int(angle), int(grade)

        if not (20 <= angle <= 60):
            return False, f"Invalid angle: {angle}"
        if not (13 <= grade <= 27):
            return False, f"Invalid grade: {grade}"

        holds = holds_str.split()
        num_start = sum(1 for h in holds if h.startswith("start"))
        num_finish = sum(1 for h in holds if h.startswith("finish"))

        if not (1 <= num_start <= 2):
            return False, f"Must have 1-2 start holds, got {num_start}"
        if not (1 <= num_finish <= 2):
            return False, f"Must have 1-2 finish holds, got {num_finish}"
        if len(holds) >= 25:
            return False, f"Too many holds: {len(holds)}"

        return True, "Valid"


In [19]:
class KilterTrainer(Trainer):
    """
    Custom Trainer that handles:
    - Training: Custom prefix/remaining format with set-based loss
    - Evaluation: Standard input_ids format with standard GPT loss
    """

    def __init__(self, eval_collator=None, **kwargs):
        super().__init__(**kwargs)
        self.eval_collator = eval_collator

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
        ):
        """Custom loss computation that handles 'remaining' parameter for training."""
        if "remaining" in inputs:
            # Training with custom format
            remaining = inputs.pop("remaining", None)
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                remaining=remaining,
            )
            if remaining is not None:
                inputs["remaining"] = remaining
            loss = outputs.loss
        else:
            # Evaluation with standard format - use standard GPT loss
            labels = inputs.pop("labels", None)
            outputs = model.model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                labels=labels,
            )
            if labels is not None:
                inputs["labels"] = labels
            loss = outputs.loss

        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """Override prediction step for evaluation. Handles both custom and standard formats."""
        if "remaining" in inputs:
            remaining = inputs.pop("remaining", None)
            inputs = self._prepare_inputs(inputs)

            with torch.no_grad():
                outputs = model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs.get("attention_mask"),
                    remaining=remaining,
                )
                loss = outputs.loss
                logits = outputs.logits
        else:
            labels = inputs.get("labels")
            inputs = self._prepare_inputs(inputs)

            with torch.no_grad():
                outputs = model.model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs.get("attention_mask"),
                    labels=inputs.get("labels"),
                )
                loss = outputs.loss if labels is not None else None
                logits = outputs.logits

        if prediction_loss_only:
            return (loss, None, None)

        return (loss, logits, inputs.get("labels"))

    def _save(self, output_dir: Optional[str] = None, state_dict=None):
        """Override save to handle the ShuffleKilterGPT wrapper. Save the inner GPT2 model."""
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)

        if hasattr(self.model, "model"):
            self.model.model.save_pretrained(
                output_dir, state_dict=state_dict, safe_serialization=True
            )
        else:
            super()._save(output_dir, state_dict)


In [ ]:
def load_model(model_path: str, device: str = "cpu") -> ShuffleKilterGPT:
    gpt2_model = GPT2LMHeadModel.from_pretrained(model_path)
    model = ShuffleKilterGPT(vocab_size=gpt2_model.config.vocab_size)
    model.model = gpt2_model
    model.config = gpt2_model.config
    model.to(device)
    model.eval()
    return model


In [ ]:
from src.data_processing import DataPreprocessing
from src.tokenizer import train_tokenizer

# Initialize wandb
config = {
    "architecture": "ShuffleKilterGPT",
    "n_embd": 256,
    "n_head": 4,
    "n_layer": 6,
    "n_positions": 128,
    "dropout": 0.1,
    
}

wandb.init(
    project="climb-gpt-shuffle",
    name=f"set_based_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    config=config,
)

run_name = wandb.run.name
OUT_DIR = f"models/climb_gpt_shuffle/{run_name}"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load and split data
dp = DataPreprocessing()
datasets = dp.load_climbs()

train_test = datasets.train_test_split(test_size=0.2, seed=42)
val_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

datasets = {
    "train": train_test["train"],
    "val": val_test["train"],
    "test": val_test["test"],
}

# Train tokenizer
tokenizer = train_tokenizer(datasets, OUT_DIR)

# Augment datasets for set-based learning
datasets = preprocess_datasets_for_set_learning(
    datasets, tokenizer, num_augmentations=3
)

wandb.config.update(
    {
        "train_size_augmented": len(datasets["train"]),
        "val_size_augmented": len(datasets["val"]),
        "test_size_augmented": len(datasets["test"]),
        "vocab_size": tokenizer.vocab_size,
    }
)

# Create model
model = ShuffleKilterGPT(
    vocab_size=tokenizer.vocab_size,
    n_embd=config["n_embd"],
    n_head=config["n_head"],
    n_layer=config["n_layer"],
    n_positions=config["n_positions"],
    dropout=config["dropout"],
)

wandb.watch(model.model, log="all", log_freq=1000)

# Create data collators
train_collator = SetBasedDataCollator(tokenizer=tokenizer)
eval_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁
train/global_step,▁
eval/runtime,22.5244
eval/samples_per_second,341.808
eval/steps_per_second,42.754
train/epoch,0.00283
train/global_step,30


Loaded 76992 routes from cache data/climbs_cleaned.csv
Built vocabulary with 1932 tokens (1928 holds)

Vocab size: 1932 tokens
First 10 tokens: [('hand1144', 314), ('finish1197', 527), ('finish1250', 739), ('start1490', 1493), ('start1330', 1057), ('hand1359', 1174), ('feet1224', 632), ('feet1136', 280), ('start1161', 381), ('feet1240', 696)]

Sample encodings:

Input: angle35_grade14_feet1595_start1400
Tokens: ['[BOS]', 'angle35', 'grade14', 'feet1595', '[UNK]', '[EOS]', ('[PAD]', 19)]

Input: angle40_grade15_feet1595_start1596_hand1597_finish1598
Tokens: ['[BOS]', 'angle40', 'grade15', 'feet1595', 'start1596', 'hand1597', 'finish1598', '[EOS]', ('[PAD]', 17)]
Saving tokenizer to models/climb_gpt_set/set_based_20251103_000912


In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    overwrite_output_dir=True,
    logging_steps=100,
    eval_steps=500,
    save_steps=500,
    num_train_epochs=30,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    report_to="wandb",
    remove_unused_columns=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir=f"{OUT_DIR}/logs",
    load_best_model_at_end=True,
    dataloader_pin_memory=False,
    run_name=run_name,
    )


# Create trainer
trainer = KilterTrainer(
    model=model,
    args=training_args,
    data_collator=train_collator,
    eval_collator=eval_collator,
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5,
        )
    ],
)

# Train
trainer.train()

# Save model
model.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# Log model as artifact
artifact = wandb.Artifact(
    name=f"climb-gpt-shuffle-{run_name}",
    type="model",
    description="Set-based KilterGPT model",
)
artifact.add_dir(OUT_DIR)
wandb.log_artifact(artifact)

# Evaluate on test set
test_results = trainer.evaluate(datasets["test"])
wandb.log({"test/loss": test_results["eval_loss"]})
wandb.summary["final_test_loss"] = test_results["eval_loss"]
wandb.summary["model_path"] = OUT_DIR

# Generate sample routes
model.to(device)
test_configs = [
    {"angle": 40, "grade": 18},
    {"angle": 30, "grade": 15},
    {"angle": 50, "grade": 22},
]

print("\nSample Generated Routes:")
for config in test_configs:
    print(f"\nangle={config['angle']}, grade={config['grade']}")
    for i in range(3):
        route = model.generate_route(
            tokenizer=tokenizer,
            angle=config["angle"],
            grade=config["grade"],
            temperature=0.9,
            device=device,
        )
        is_valid, msg = model.validate_route(route)
        status = "✓" if is_valid else f"✗ ({msg})"
        print(f"  {i+1}. {route} [{status}]")

wandb.finish()


In [ ]:
# Generate sample routes
model.to(device)
test_configs = [
    {"angle": 40, "grade": 18},
    {"angle": 30, "grade": 15},
    {"angle": 50, "grade": 22},
]

print("\nSample Generated Routes:")
for config in test_configs:
    print(f"\nangle={config['angle']}, grade={config['grade']}")
    for i in range(3):
        route = model.generate_route(
            tokenizer=tokenizer,
            angle=config["angle"],
            grade=config["grade"],
            temperature=0.9,
            device=device,
        )
        is_valid, msg = model.validate_route(route)
        status = "✓" if is_valid else f"✗ ({msg})"
        print(f"  {i+1}. {route} [{status}]")

wandb.finish()
